In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp

silver_table = DeltaTable.forName(spark,"workspace.default.sales_silver")

bronze_updates_df = spark.table("workspace.default.sales_bronze")

clean_updates_df = bronze_updates_df \
    .filter("user_id is not null and amount > 0") \
    .dropDuplicates(["transaction_id"])

silver_table.alias("target").merge(
    source = clean_updates_df.alias("s"),
    condition = "target.transaction_id = s.transaction_id"
) \
    .whenMatchedUpdate(set = {
        "user_id":"s.user_id",
        "amount":"s.amount" ,
        "event_time":"s.event_time",
        "ingest_time":"current_timestamp()"

    }) \
        .whenNotMatchedInsert(values = {
            "transaction_id":"s.transaction_id",
            "user_id":"s.user_id",
            "amount":"s.amount",
            "event_time":"s.event_time",
            "ingest_time":"current_timestamp()"
        }) \
            .execute()

display(spark.sql("select * from workspace.default.sales_silver").orderBy("user_id"))